In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder.appName("cache_practice").master("spark://spark-master:7077").getOrCreate()

In [2]:
orders_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/data/orders_1M.csv")  
)

orders_df.show(5)

+--------+-----------+------+------------+-------+-----+----------+
|order_id|customer_id|amount|order_status|country|state|order_date|
+--------+-----------+------+------------+-------+-----+----------+
|       1|       1861|706.08|     PENDING|     UK|   KA|2022-09-18|
|       2|       8236|473.24|   DELIVERED| Canada|   MH|2023-06-29|
|       3|       4230| 773.3|     SHIPPED| Canada|   DL|2021-05-31|
|       4|       5656|311.33|     PENDING|  India|   DL|2022-12-19|
|       5|       4936|445.11|     PENDING| Canada|   DL|2021-03-09|
+--------+-----------+------+------------+-------+-----+----------+
only showing top 5 rows



In [3]:
orders_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- order_status: string (nullable = true)
 |-- country: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: date (nullable = true)



In [4]:
df = (
        orders_df.filter(F.col("order_date") > '2021-01-01')
        .withColumn("customer_status", F.col("customer_id") + F.col("amount"))
        .groupBy("state")
        .agg(
            F.sum("amount").alias("total_amount")
        )
)
df.show()

+-----+-------------------+
|state|       total_amount|
+-----+-------------------+
|   DL|7.628928760000002E7|
|   TN|7.641954253999989E7|
|   UP| 7.64327678400003E7|
|   MH|7.624339629999948E7|
|   KA|7.644903697999969E7|
+-----+-------------------+



In [5]:
import time

start = time.time()
print(df.count())
print("First run time:", time.time() - start)

start = time.time()
print(df.count())
print("Second run time:", time.time() - start)

5
First run time: 4.11013650894165
5
Second run time: 2.883254051208496


In [6]:
df.cache()

DataFrame[state: string, total_amount: double]

In [7]:
import time

# First run (this will populate cache)
start = time.time()
print(df.count())
print("First run (with cache, but materializing):", time.time() - start)

# Second run (should be fast)
start = time.time()
print(df.count())
print("Second run (cached):", time.time() - start)

5
First run (with cache, but materializing): 4.0471351146698
5
Second run (cached): 0.18097925186157227


In [9]:
from pyspark import StorageLevel
df.persist(StorageLevel.MEMORY_AND_DISK)

DataFrame[state: string, total_amount: double]

In [10]:

import time

# First run (this will populate cache)
start = time.time()
print(df.count())
print("First run (with cache, but materializing):", time.time() - start)

# Second run (should be fast)
start = time.time()
print(df.count())
print("Second run (cached):", time.time() - start)

5
First run (with cache, but materializing): 0.16889715194702148
5
Second run (cached): 0.15626215934753418
